# Tu modelo en la web: plantilla

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel01010101010101/Ejercicio_ejemplo_machine_learning/blob/main/notebooks/plantilla_tu_modelo.ipynb)

Con este cuaderno conviertes **tus datos** (un archivo CSV) en un modelo que funciona en una página web, como el [Predictor Saber 11](https://daniel01010101010101.github.io/Ejercicio_ejemplo_machine_learning/).

**Solo tienes que hacer tres cosas:**

1. Cambiar la celda **«1. Configura»** (el nombre de tu página, tu nombre, tus datos y qué quieres predecir).
2. Menú **Entorno de ejecución → Ejecutar todas** (en inglés: *Runtime → Run all*).
3. Subir el archivo `modelo.js` que se descarga al final a la carpeta `web/` de tu repositorio en GitHub.

Si no cambias nada, el cuaderno usa un ejemplo (precios de diamantes) para que veas cómo funciona.

**Qué debe tener tu CSV:** una fila por caso, una columna **numérica** que quieras predecir y entre 3 y 10 columnas que la expliquen (texto o números). No uses datos personales (nombres, documentos, correos) ni columnas como género o etnia para predecir.

## 1. Configura (lo único que cambias)

Cambia los textos entre comillas. Los nombres de columnas deben ser **exactamente** los de tu CSV (mayúsculas, tildes y espacios incluidos).

In [1]:
# ============================================================
# 1. CONFIGURA AQUÍ: es lo único que tienes que cambiar
# ============================================================

# --- Tu página ---
TITULO = "Predictor de precio de diamantes"   # nombre de tu página
PREGUNTA = "¿Cuánto vale un diamante?"         # la frase grande de la portada
AUTOR = "Tu nombre"                             # aparece al pie de la página
CURSO = "Machine Learning 1 · Universidad EAN"
USUARIO_GITHUB = ""                             # opcional: tu usuario de GitHub, para el enlace del pie

# --- Tus datos ---
# Pega el enlace a tu CSV, o deja URL_DATOS = None para subir el archivo desde tu computador.
URL_DATOS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
NOMBRE_DATOS = "Diamonds (seaborn-data)"        # cómo se llaman tus datos (aparece en la página)
URL_FUENTE = "https://github.com/mwaskom/seaborn-data"   # de dónde salieron (opcional)

# --- Qué quieres predecir ---
OBJETIVO = "price"            # nombre EXACTO de la columna numérica a predecir
ETIQUETA_OBJETIVO = "Precio"  # cómo se llama en la página
UNIDAD = "USD"                # puntos, pesos, %, USD, horas...

# --- Con qué lo predices (las preguntas del formulario) ---
VARIABLES = ["carat", "cut", "color", "clarity"]   # entre 3 y 10 columnas; None = el cuaderno las elige
EXCLUIR = []                  # columnas que NO se deben usar (identificadores, nombres, género...)

# --- Opcional: textos más claros para el formulario (puedes dejarlos vacíos: {}) ---
ETIQUETAS = {                 # columna -> pregunta del formulario
    "carat": "Peso en quilates",
    "cut": "Calidad del corte",
    "color": "Color (D es el mejor, J el peor)",
    "clarity": "Pureza",
}
ETIQUETAS_CATEGORIAS = {      # columna -> {valor en los datos: texto en el formulario}
    "cut": {"Fair": "Regular", "Good": "Bueno", "Very Good": "Muy bueno", "Premium": "Premium", "Ideal": "Ideal"},
}
ORDEN_CATEGORIAS = {          # columna -> orden de las opciones
    "cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
}
AVISO = "Es una estimación con error: úsala para aprender y discutir, no para tomar decisiones importantes."

## 2. Carga los datos

Si pusiste un enlace, se descarga solo. Si dejaste `URL_DATOS = None`, aparece un botón para elegir tu archivo. El cuaderno reconoce CSV separados por coma o por punto y coma.

In [2]:
import datetime as dt
import io
import json
import math
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

EN_COLAB = "google.colab" in sys.modules
SIN_INFO = "Sin información"
OTRA = "Otra (pocos casos)"
SEMILLA = 42
CARPETA_SALIDA = Path("mi_modelo")


def fmt(numero, decimales=0):
    """Formato colombiano: 12.345,6"""
    texto = f"{numero:,.{decimales}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def leer_csv(fuente):
    """Lee un CSV separado por comas o por punto y coma, en UTF-8 o Latin-1."""
    for codificacion in ("utf-8", "latin-1"):
        try:
            origen = io.BytesIO(fuente) if isinstance(fuente, bytes) else fuente
            return pd.read_csv(origen, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
    raise ValueError("No pude leer el archivo. Guárdalo como «CSV UTF-8» y vuelve a intentar.")


if URL_DATOS:
    datos = leer_csv(URL_DATOS)
else:
    from google.colab import files
    print("Elige tu archivo CSV (botón «Elegir archivos»):")
    subido = files.upload()
    datos = leer_csv(next(iter(subido.values())))
datos.columns = [str(c).strip() for c in datos.columns]
print(f"Tus datos tienen {fmt(len(datos))} filas y {datos.shape[1]} columnas:")
print(", ".join(datos.columns))
datos.head()

Tus datos tienen 53.940 filas y 10 columnas:
carat, cut, color, clarity, depth, table, price, x, y, z


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## 3. Revisa los datos y lo que quieres predecir

La tabla muestra cada columna: si es número o texto, cuántos vacíos tiene y cuántos valores distintos. Si el objetivo es sí/no (0 y 1), el modelo estimará un porcentaje.

In [3]:
def a_numero(serie):
    """Convierte a número; acepta coma decimal (3,5)."""
    return pd.to_numeric(serie.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def es_numerica(serie):
    return a_numero(serie).notna().mean() >= 0.95 * serie.notna().mean()


resumen = pd.DataFrame({
    "tipo": ["número" if es_numerica(datos[c]) else "texto" for c in datos.columns],
    "% vacíos": (100 * datos.isna().mean()).round(1).values,
    "valores distintos": datos.nunique().values,
    "ejemplo": [datos[c].dropna().iloc[0] if datos[c].notna().any() else "" for c in datos.columns],
}, index=datos.columns)
display(resumen)

if OBJETIVO not in datos.columns:
    raise ValueError(f"No encuentro la columna OBJETIVO = {OBJETIVO!r}. Copia el nombre exacto de la tabla de arriba.")
y_todo = a_numero(datos[OBJETIVO])
if y_todo.notna().mean() < 0.5:
    raise ValueError(f"La columna {OBJETIVO!r} no es numérica. Elige una columna de números para predecir.")
ES_SI_NO = set(y_todo.dropna().unique()) <= {0, 1}
if ES_SI_NO:
    y_todo = 100 * y_todo
    UNIDAD = "%"
    print("Tu objetivo es sí/no (0 y 1): el modelo estimará un porcentaje de 0 a 100.")
print(f"Objetivo «{OBJETIVO}»: promedio {fmt(y_todo.mean(), 1)} · mínimo {fmt(y_todo.min(), 1)} · "
      f"máximo {fmt(y_todo.max(), 1)} · vacíos {fmt(y_todo.isna().sum())}")

,tipo,% vacíos,valores distintos,ejemplo
carat,número,0.0,273,0.23
cut,texto,0.0,5,Ideal
color,texto,0.0,7,E
clarity,texto,0.0,8,SI2
depth,número,0.0,184,61.5
table,número,0.0,127,55.0
price,número,0.0,11602,326
x,número,0.0,554,3.95
y,número,0.0,552,3.98
z,número,0.0,375,2.43


Objetivo «price»: promedio 3.932,8 · mínimo 326,0 · máximo 18.823,0 · vacíos 0


## 4. Separa entrenamiento y prueba

El 80 % de las filas es para entrenar y el 20 % para evaluar con casos que el modelo nunca vio. Se separa **antes** de aprender cualquier cosa de los datos.

In [4]:
filas_validas = y_todo.notna()
x_todo = datos.loc[filas_validas].drop(columns=[OBJETIVO])
y_todo = y_todo[filas_validas]
x_train, x_test, y_train, y_test = train_test_split(x_todo, y_todo, test_size=0.2, random_state=SEMILLA)
MIN_CASOS = max(20, round(0.005 * len(x_train)))
print(f"Entrenamiento: {fmt(len(x_train))} filas · Prueba: {fmt(len(x_test))} filas")
print(f"Una categoría con menos de {MIN_CASOS} casos en entrenamiento se agrupa en «{OTRA}».")

Entrenamiento: 43.152 filas · Prueba: 10.788 filas
Una categoría con menos de 216 casos en entrenamiento se agrupa en «Otra (pocos casos)».


## 5. Prepara las preguntas del formulario

Todo se aprende **solo con entrenamiento** y se aplica igual a la prueba:

- Una columna de **números** se parte en 5 rangos del mismo tamaño (por ejemplo, «0,40 a 0,70»).
- Una columna de **texto** conserva sus categorías; las que tienen muy pocos casos se agrupan y los vacíos pasan a «Sin información».

Así toda la página funciona con sumas: cada respuesta tiene un número que suma o resta.

In [5]:
def como_texto(serie):
    """Texto limpio de una columna; los números se escriben sin decimales de sobra."""
    if es_numerica(serie):
        numeros = a_numero(serie)
        texto = numeros.map(lambda v: "" if pd.isna(v) else fmt(v, 0 if float(v).is_integer() else 2))
    else:
        texto = serie.astype("string").str.strip()
    return texto.fillna(SIN_INFO).replace("", SIN_INFO).astype(object)


def aprender_preparacion(train, columna):
    """Aprende, SOLO con train, cómo convertir una columna en categorías."""
    serie = train[columna]
    if es_numerica(serie) and serie.nunique() > 10:
        valores = a_numero(serie).dropna()
        cortes = sorted(set(float(c) for c in np.quantile(valores, [0.2, 0.4, 0.6, 0.8]).round(6)))
        return {"tipo": "rangos", "cortes": cortes,
                "decimales": 0 if all(c.is_integer() for c in cortes) else 2}
    conteo = como_texto(serie).value_counts()
    return {"tipo": "categorias", "frecuentes": list(conteo[conteo >= MIN_CASOS].index[:29])}


def etiquetas_rangos(prep):
    c, d = prep["cortes"], prep["decimales"]
    if not c:
        return ["Todos"]
    return ([f"hasta {fmt(c[0], d)}"] + [f"{fmt(a, d)} a {fmt(b, d)}" for a, b in zip(c[:-1], c[1:])]
            + [f"más de {fmt(c[-1], d)}"])


def aplicar_preparacion(tabla, columna, prep):
    if prep["tipo"] == "rangos":
        bordes = [-np.inf] + prep["cortes"] + [np.inf]
        rangos = pd.cut(a_numero(tabla[columna]), bins=bordes, labels=etiquetas_rangos(prep))
        return rangos.astype(object).where(rangos.notna(), SIN_INFO)
    texto = como_texto(tabla[columna])
    return texto.where(texto.isin(prep["frecuentes"]), OTRA)


if VARIABLES is None:
    # El cuaderno propone las columnas que más explican el objetivo por sí solas (medido en train).
    puntajes = {}
    for columna in [c for c in x_train.columns if c not in EXCLUIR]:
        categoria = aplicar_preparacion(x_train, columna, aprender_preparacion(x_train, columna))
        if categoria.nunique() < 2:
            continue
        medias = y_train.groupby(categoria).transform("mean")
        puntajes[columna] = 1 - ((y_train - medias) ** 2).sum() / ((y_train - y_train.mean()) ** 2).sum()
    VARIABLES = [c for c, _ in sorted(puntajes.items(), key=lambda par: -par[1])[:8]]
    print("El cuaderno eligió estas columnas:", VARIABLES)

faltan = [c for c in VARIABLES if c not in x_train.columns]
if faltan:
    raise ValueError(f"Estas columnas de VARIABLES no existen en tus datos: {faltan}")
if any(c in EXCLUIR for c in VARIABLES):
    raise ValueError("Una columna está a la vez en VARIABLES y en EXCLUIR.")

preparacion = {c: aprender_preparacion(x_train, c) for c in VARIABLES}   # SOLO con train
x_train_cat = pd.DataFrame({c: aplicar_preparacion(x_train, c, preparacion[c]) for c in VARIABLES}, index=x_train.index)
x_test_cat = pd.DataFrame({c: aplicar_preparacion(x_test, c, preparacion[c]) for c in VARIABLES}, index=x_test.index)
for c in VARIABLES:
    p = preparacion[c]
    detalle = ("rangos: " + ", ".join(etiquetas_rangos(p))) if p["tipo"] == "rangos" else f"{x_train_cat[c].nunique()} categorías"
    print(f"  {ETIQUETAS.get(c, c)} ({c}): {detalle}")

  Peso en quilates (carat): rangos: hasta 0,35, 0,35 a 0,54, 0,54 a 0,90, 0,90 a 1,13, más de 1,13
  Calidad del corte (cut): 5 categorías
  Color (D es el mejor, J el peor) (color): 7 categorías
  Pureza (clarity): 8 categorías


## 6. Entrena el modelo y compáralo con una línea base

El modelo es el mismo del Predictor Saber 11: `OneHotEncoder` + `Ridge`. La **línea base** responde siempre el promedio; tu modelo vale la pena en la medida en que le gana.

- **MAE**: en promedio, ¿por cuánto se equivoca? (en la unidad de tu objetivo)
- **R²**: qué parte de la variación explica (0 = igual que adivinar el promedio).

In [6]:
modelo = Pipeline([
    ("preprocesamiento", ColumnTransformer([("categoricas", OneHotEncoder(handle_unknown="ignore"), VARIABLES)])),
    ("regresion", Ridge(alpha=1.0)),
]).fit(x_train_cat, y_train)
base = DummyRegressor(strategy="mean").fit(x_train_cat, y_train)


def metricas(y_real, y_pred):
    return {"mae": float(mean_absolute_error(y_real, y_pred)),
            "rmse": float(math.sqrt(mean_squared_error(y_real, y_pred))),
            "r2": float(r2_score(y_real, y_pred))}


pred = modelo.predict(x_test_cat)
resultados = {
    "modelo": {"nombre": "Regresión lineal (Ridge)", **metricas(y_test, pred)},
    "linea_base": {"nombre": "Línea base: siempre el promedio", **metricas(y_test, base.predict(x_test_cat))},
}
display(pd.DataFrame(resultados.values()).set_index("nombre").round(3))
mae = resultados["modelo"]["mae"]
mejora = 1 - mae / resultados["linea_base"]["mae"]
print(f"Tu modelo se equivoca en promedio {fmt(mae, 1)} {UNIDAD}; la línea base, "
      f"{fmt(resultados['linea_base']['mae'], 1)} {UNIDAD}. Mejora: {fmt(100 * mejora)} %.")
if mejora < 0.05:
    print("Ojo: casi no le gana a la línea base. Prueba con otras columnas en VARIABLES.")

,mae,rmse,r2
nombre,,,
Regresión lineal (Ridge),1103.029,1857.524,0.783
Línea base: siempre el promedio,3020.506,3987.222,-0.000


Tu modelo se equivoca en promedio 1.103,0 USD; la línea base, 3.020,5 USD. Mejora: 63 %.


## 7. Qué suma y qué resta cada respuesta

Para cada pregunta: qué pasa al cambiar la respuesta más común por la que más suma y por la que más resta, **dejando lo demás igual**. Son asociaciones, no causas.

In [7]:
codificador = modelo.named_steps["preprocesamiento"].named_transformers_["categoricas"]
coeficientes = modelo.named_steps["regresion"].coef_
inicio = 0
for i, v in enumerate(VARIABLES):
    categorias = list(codificador.categories_[i])
    coefs = dict(zip(categorias, coeficientes[inicio:inicio + len(categorias)]))
    inicio += len(categorias)
    conteo = x_train_cat[v].value_counts()
    ref = conteo.idxmax()
    otras = {c: k - coefs[ref] for c, k in coefs.items() if c != ref and conteo.get(c, 0) >= MIN_CASOS}
    if not otras:
        continue
    alta, baja = max(otras, key=otras.get), min(otras, key=otras.get)
    nombre = ETIQUETAS_CATEGORIAS.get(v, {})
    texto = (f"{ETIQUETAS.get(v, v)}: pasar de «{nombre.get(ref, ref)}» a «{nombre.get(alta, alta)}» "
             f"{'suma' if otras[alta] >= 0 else 'resta'} {fmt(abs(otras[alta]), 1)} {UNIDAD}")
    if baja != alta:
        texto += f"; a «{nombre.get(baja, baja)}», {'suma' if otras[baja] >= 0 else 'resta'} {fmt(abs(otras[baja]), 1)}"
    print(texto + ".")

Peso en quilates: pasar de «0,54 a 0,90» a «más de 1,13» suma 7.895,0 USD; a «hasta 0,35», resta 2.414,4.
Calidad del corte: pasar de «Ideal» a «Muy bueno» resta 43,5 USD; a «Regular», resta 461,9.
Color (D es el mejor, J el peor): pasar de «G» a «D» suma 407,9 USD; a «J», resta 1.282,2.
Pureza: pasar de «SI1» a «IF» suma 1.360,9 USD; a «I1», resta 2.789,1.


## 8. Exporta `modelo.js` y verifica

Guardamos el modelo como **números** (el intercepto y un coeficiente por respuesta) dentro de `modelo.js`, junto con los textos de tu página. Después, una **prueba de paridad** recalcula las predicciones con JavaScript y exige que den lo mismo que scikit-learn (tolerancia 1e-6).

In [8]:
def etiqueta_de(v, valor):
    return str(ETIQUETAS_CATEGORIAS.get(v, {}).get(valor, valor))


def orden_de(v, valor):
    """Orden de las opciones: el de ORDEN_CATEGORIAS, los rangos de menor a mayor, y al final los vacíos."""
    if valor == SIN_INFO:
        return (2, 0, "")
    if valor == OTRA:
        return (3, 0, "")
    if preparacion[v]["tipo"] == "rangos":
        return (0, etiquetas_rangos(preparacion[v]).index(valor), "")
    orden = [str(o) for o in ORDEN_CATEGORIAS.get(v, [])]
    if valor in orden:
        return (0, orden.index(valor), "")
    numero = pd.to_numeric(valor.replace(".", "").replace(",", "."), errors="coerce")
    return (1, float(numero) if pd.notna(numero) else math.inf, valor.lower())


def paso_redondo(valor):
    p = 10 ** math.floor(math.log10(valor))
    return next(m * p for m in (1, 2, 2.5, 5, 10) if valor <= m * p)


if ES_SI_NO:
    minimo, maximo = 0.0, 100.0
else:
    paso = paso_redondo((y_todo.max() - min(0, y_todo.min())) / 5)
    minimo = 0.0 if y_todo.min() >= 0 else math.floor(y_todo.min() / paso) * paso
    maximo = math.ceil(y_todo.max() / paso) * paso

variables = []
inicio = 0
for i, v in enumerate(VARIABLES):
    categorias = list(codificador.categories_[i])
    coefs = coeficientes[inicio:inicio + len(categorias)]
    inicio += len(categorias)
    conteo = x_train_cat[v].value_counts()
    filas = [{"valor": str(c), "etiqueta": etiqueta_de(v, c), "coeficiente": float(k),
              "frecuencia": float(conteo.get(c, 0) / len(x_train_cat)), "n": int(conteo.get(c, 0)),
              "en_formulario": bool(conteo.get(c, 0) >= MIN_CASOS)} for c, k in zip(categorias, coefs)]
    filas.sort(key=lambda f: orden_de(v, f["valor"]))
    variables.append({"nombre": v, "etiqueta": ETIQUETAS.get(v, v), "etiqueta_corta": ETIQUETAS.get(v, v),
                      "grupo": "caso", "ayuda": "Elige el rango." if preparacion[v]["tipo"] == "rangos" else "",
                      "mas_frecuente": str(conteo.idxmax()), "categorias": filas})

# 5 casos del set de prueba (estimaciones bajas, medias y altas) que el formulario puede mostrar.
visibles = pd.Series(True, index=x_test_cat.index)
for v in VARIABLES:
    conteo = x_train_cat[v].value_counts()
    visibles &= x_test_cat[v].isin(conteo[conteo >= MIN_CASOS].index)
x_casos, y_casos = x_test_cat[visibles], y_test[visibles]
orden = np.argsort(modelo.predict(x_casos), kind="stable")
casos = []
for numero, q in enumerate(np.linspace(0.05, 0.95, 5), start=1):
    i = int(orden[int(round(q * (len(orden) - 1)))])
    fila = x_casos.iloc[[i]]
    casos.append({"id": numero, "entradas": {v: str(fila.iloc[0][v]) for v in VARIABLES},
                  "puntaje_real": float(y_casos.iloc[i]), "prediccion_sklearn": float(modelo.predict(fila)[0])})

ridge = modelo.named_steps["regresion"]
cobertura = float(np.mean(np.abs(y_test.to_numpy() - pred) <= mae))
exportado = {
    "version": "1.0.0",
    "proyecto": TITULO,
    "fuente": {"nombre": NOMBRE_DATOS, "url": URL_FUENTE or URL_DATOS or "", "portal": "", "id": ""},
    "fecha_entrenamiento": dt.date.today().isoformat(),
    "n_total": int(len(y_todo)), "n_entrenamiento": int(len(y_train)), "n_prueba": int(len(y_test)),
    "semilla": SEMILLA, "min_casos": MIN_CASOS,
    "algoritmo": "Ridge (alpha = 1) con codificación one-hot",
    "sklearn_version": sklearn.__version__,
    "objetivo": {"nombre": OBJETIVO, "etiqueta": ETIQUETA_OBJETIVO, "minimo": minimo, "maximo": maximo, "unidad": UNIDAD},
    "metricas": resultados,
    "cobertura_mae": cobertura,
    "promedio_nacional": float(y_todo.mean()),
    "promedio_entrenamiento": float(y_train.mean()),
    "intercepto": float(ridge.intercept_),
    "variables": variables,
    "excluidas": [{"nombre": c, "motivo": "La excluyó el autor del modelo."} for c in EXCLUIR],
    "casos_prueba": casos,
    "textos": {
        "titulo": TITULO, "subtitulo": CURSO, "pregunta": PREGUNTA, "aviso_etico": AVISO,
        "etiqueta_promedio": "Promedio de los datos", "subetiqueta_medidor": UNIDAD,
        "autor": AUTOR, "autor_url": f"https://github.com/{USUARIO_GITHUB}" if USUARIO_GITHUB else "",
        "grupos": {"caso": "Datos del caso"},
    },
}
# Identidad que usa la gráfica de aportes: intercepto + suma(frecuencia x coeficiente) = promedio de train.
reconstruido = exportado["intercepto"] + sum(c["frecuencia"] * c["coeficiente"] for v in variables for c in v["categorias"])
assert abs(reconstruido - exportado["promedio_entrenamiento"]) < 1e-6

CARPETA_SALIDA.mkdir(exist_ok=True)
ruta_js = CARPETA_SALIDA / "modelo.js"
texto = json.dumps(exportado, ensure_ascii=False, indent=2)
ruta_js.write_text("// Generado por el cuaderno plantilla_tu_modelo.ipynb. No lo edites a mano.\n"
                   f"window.MODELO = {texto};\n", encoding="utf-8")
print(f"Listo: {ruta_js} ({fmt(ruta_js.stat().st_size / 1024, 1)} KB)")

Listo: mi_modelo/modelo.js (9,2 KB)


In [9]:
PRUEBA_PARIDAD_JS = r"""#!/usr/bin/env node
/*
 * Prueba de paridad: la predicción en JavaScript debe coincidir con scikit-learn.
 *
 * Uso:  node scripts/prueba_paridad.js [web/modelo.js] [casos_extra.json]
 *
 * 1. Carga modelo.js igual que el navegador (el archivo define window.MODELO).
 * 2. Recalcula cada caso con la misma fórmula de web/index.html:
 *    intercepto + suma de los coeficientes de las categorías elegidas.
 * 3. Compara con la predicción de scikit-learn guardada al exportar.
 *
 * Termina con código 1 si alguna diferencia supera la tolerancia (1e-6).
 */
'use strict';

var fs = require('fs');
var path = require('path');
var vm = require('vm');

var TOLERANCIA = 1e-6;
var rutaModelo = process.argv[2] || path.join(__dirname, '..', 'web', 'modelo.js');
var rutaExtra = process.argv[3];

function cargarModelo(ruta) {
  var contexto = { window: {} };
  vm.createContext(contexto);
  vm.runInContext(fs.readFileSync(ruta, 'utf8'), contexto, { filename: ruta });
  if (!contexto.window.MODELO) {
    throw new Error(ruta + ' no definió window.MODELO');
  }
  return contexto.window.MODELO;
}

// Misma función que usa web/index.html.
function predecir(modelo, respuestas) {
  var total = modelo.intercepto;
  modelo.variables.forEach(function (variable) {
    var valor = respuestas[variable.nombre];
    var categoria = variable.categorias.find(function (c) { return c.valor === valor; });
    if (categoria) total += categoria.coeficiente; // categoría desconocida: aporta 0
  });
  return total;
}

function comparar(modelo, casos) {
  var resumen = { n: casos.length, maxDif: 0, fallos: 0, desconocidas: 0 };
  casos.forEach(function (caso) {
    modelo.variables.forEach(function (variable) {
      var valor = caso.entradas[variable.nombre];
      var existe = variable.categorias.some(function (c) { return c.valor === valor; });
      if (!existe) resumen.desconocidas += 1;
    });
    var dif = Math.abs(predecir(modelo, caso.entradas) - caso.prediccion_sklearn);
    if (dif > resumen.maxDif) resumen.maxDif = dif;
    if (!(dif <= TOLERANCIA)) resumen.fallos += 1;
  });
  return resumen;
}

function formato(numero) {
  return numero.toFixed(6).padStart(12);
}

var modelo = cargarModelo(rutaModelo);
var ok = true;

console.log('Modelo ' + modelo.version + (modelo.periodo ? ' · periodo ' + modelo.periodo : '') +
            ' · ' + modelo.variables.length + ' variables');
console.log('Caso       sklearn   JavaScript    diferencia');
modelo.casos_prueba.forEach(function (caso) {
  var js = predecir(modelo, caso.entradas);
  var dif = Math.abs(js - caso.prediccion_sklearn);
  console.log(String(caso.id).padStart(4) + ' ' + formato(caso.prediccion_sklearn) + ' ' +
              formato(js) + '  ' + dif.toExponential(2));
});

var casos = comparar(modelo, modelo.casos_prueba);
console.log('Casos de modelo.js: ' + casos.n + ', diferencia máxima ' +
            casos.maxDif.toExponential(2) + ', fuera de tolerancia: ' + casos.fallos);
if (casos.fallos > 0 || casos.desconocidas > 0) ok = false;

if (rutaExtra) {
  var extra = comparar(modelo, JSON.parse(fs.readFileSync(rutaExtra, 'utf8')));
  console.log('Casos extra del set de prueba: ' + extra.n + ', diferencia máxima ' +
              extra.maxDif.toExponential(2) + ', fuera de tolerancia: ' + extra.fallos);
  if (extra.fallos > 0) ok = false;
}

// La gráfica de contribuciones usa: intercepto + suma(frecuencia x coeficiente)
// = promedio del entrenamiento. Si esto falla, la gráfica no cuadraría.
var base = modelo.intercepto;
modelo.variables.forEach(function (variable) {
  variable.categorias.forEach(function (c) { base += c.frecuencia * c.coeficiente; });
});
var difBase = Math.abs(base - modelo.promedio_entrenamiento);
console.log('Identidad del promedio: diferencia ' + difBase.toExponential(2));
if (!(difBase <= TOLERANCIA)) ok = false;

console.log(ok ? 'OK: JavaScript y scikit-learn coinciden (tolerancia 1e-6).'
               : 'ERROR: la paridad falló.');
process.exit(ok ? 0 : 1);
"""


ruta_prueba = CARPETA_SALIDA / "prueba_paridad.js"
ruta_prueba.write_text(PRUEBA_PARIDAD_JS, encoding="utf-8")

node = shutil.which("node")
if node is None and EN_COLAB:
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y nodejs > /dev/null", shell=True)
    node = shutil.which("node")
if node:
    salida = subprocess.run([node, str(ruta_prueba), str(ruta_js)], capture_output=True, text=True)
    print(salida.stdout + salida.stderr)
    assert salida.returncode == 0, "La paridad falló: no subas este archivo."
else:
    # Respaldo si no hay Node: la misma cuenta en Python con los números exportados.
    for caso in casos:
        total = exportado["intercepto"] + sum(next((c["coeficiente"] for c in v["categorias"]
                                                    if c["valor"] == caso["entradas"][v["nombre"]]), 0.0)
                                              for v in variables)
        assert abs(total - caso["prediccion_sklearn"]) <= 1e-6
    print("OK: los números exportados reproducen a scikit-learn (tolerancia 1e-6).")

Modelo 1.0.0 · 4 variables
Caso       sklearn   JavaScript    diferencia
   1   213.987505   213.987505  2.84e-13
   2  1241.093663  1241.093663  0.00e+0
   3  2611.626083  2611.626083  0.00e+0
   4  5565.836933  5565.836933  0.00e+0
   5 10589.632219 10589.632219  0.00e+0
Casos de modelo.js: 5, diferencia máxima 2.84e-13, fuera de tolerancia: 0
Identidad del promedio: diferencia 4.55e-13
OK: JavaScript y scikit-learn coinciden (tolerancia 1e-6).



## 9. Descarga `modelo.js` y súbelo a tu página

La siguiente celda descarga `modelo.js`. Luego, en GitHub:

1. Entra a tu repositorio y abre la carpeta **`web`**.
2. **Add file → Upload files** y arrastra `modelo.js`.
3. **Commit changes**.
4. Espera a que la pestaña **Actions** muestre la marca verde (uno o dos minutos) y abre tu página.

Si el navegador guardó el archivo como `modelo (1).js`, **cámbiale el nombre a `modelo.js`** antes de subirlo: la página solo lee ese nombre.

In [10]:
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_js))
    print("Se descargó modelo.js. Si quedó como «modelo (1).js», cámbiale el nombre a modelo.js antes de subirlo.")
else:
    print(f"Fuera de Colab no hay descarga automática: el archivo quedó en {ruta_js}")

Fuera de Colab no hay descarga automática: el archivo quedó en mi_modelo/modelo.js
